# Interim: per-video smoothing ablation

Ports the (sigma, delta) grid-search from Paper A cells 62-66 to our
CREMA-D fusion checkpoints. At inference time, predictions on V/A and
on EXPR probabilities are replaced by a per-video Gaussian-weighted
average over the surrounding +/- delta frames. AU predictions are
NOT smoothed (cell 66 of Paper A showed it hurts AU F1).

Because CREMA-D's Russell-anchored VA labels are clip-constant, large
sigma (wide kernel) should help in the limit; the point of the
ablation is to quantify the effect size and to identify which
variants benefit most.

Output: `results/interim/ablation_smoothing.md` plus an appended
section of `ablations_summary.md`.

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("cwd:", Path.cwd())

import numpy as np
import pandas as pd
import torch
from omegaconf import OmegaConf
from torch.utils.data import DataLoader

from src.datasets.bimodal import BimodalFrameDataset, BimodalWindowDataset
from src.fusion.base import FusionConfig
from src.train_fusion import build_model, wants_window
from src.smoothing import gaussian_smooth_per_video
from src.utils.metrics import CCC_score, metric_for_Exp, p_mtl

CONFIGS = ROOT / "configs" / "interim"
RESULTS = ROOT / "results" / "interim"
RESULTS.mkdir(parents=True, exist_ok=True)

cwd: C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code


In [2]:
def collect_predictions(config_path: Path, checkpoint_path: Path):
    """Run a trained variant over its val split and return per-frame tensors."""
    cfg = OmegaConf.load(config_path)
    device = cfg.train.device if torch.cuda.is_available() else "cpu"

    ck = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    variant = ck["variant"]
    fus_cfg = FusionConfig(**ck["fus_cfg"])
    model = build_model(variant, fus_cfg)
    model.load_state_dict(ck["state_dict"])
    model.to(device).eval()

    ds_cls = BimodalWindowDataset if wants_window(variant) else BimodalFrameDataset
    extra = {"window": int(cfg.fusion.get("window", 5))} if wants_window(variant) else {}
    ds = ds_cls(
        annotations_file=cfg.data.val_annotations,
        visual_cache_dir=cfg.visual.features_cache,
        audio_cache_dir=cfg.audio.aligned_dir,
        **extra,
    )
    loader = DataLoader(ds, batch_size=int(cfg.train.batch_size), shuffle=False)

    expr_logits, va_pred, au_pred = [], [], []
    y_expr, y_va, y_aus, m_expr, m_va, m_au = [], [], [], [], [], []
    with torch.no_grad():
        for batch in loader:
            e, v, a = model(
                batch["v_feat"].to(device),
                batch["v_scores"].to(device),
                batch["a_feat"].to(device),
            )
            expr_logits.append(e.cpu().numpy())
            va_pred.append(v.cpu().numpy())
            au_pred.append(a.cpu().numpy())
            y_expr.append(batch["y_expr"].numpy())
            y_va.append(batch["y_va"].numpy())
            y_aus.append(batch["y_aus"].numpy())
            m_expr.append(batch["m_expr"].numpy())
            m_va.append(batch["m_va"].numpy())
            m_au.append(batch["m_au"].numpy())

    cat = lambda xs: np.concatenate(xs, axis=0)
    return {
        "variant": variant,
        "expr_logits": cat(expr_logits),
        "va_pred":     cat(va_pred),
        "au_pred":     cat(au_pred),
        "y_expr":      cat(y_expr),
        "y_va":        cat(y_va),
        "y_aus":       cat(y_aus),
        "m_expr":      cat(m_expr),
        "m_va":        cat(m_va),
        "m_au":        cat(m_au),
        "videoname_frames": ds.videoname_frames,
    }

In [3]:
SIGMAS = [0.1, 1, 10, 50, 100, 500, 1000, 10000, 100000]
DELTAS = [1, 5, 10, 50, 100]

def _softmax(x):
    x = x - x.max(axis=-1, keepdims=True)
    np.exp(x, out=x)
    x /= x.sum(axis=-1, keepdims=True)
    return x

def evaluate_smoothing(P):
    """Grid-search (sigma, delta) over VA (joint) and EXPR probabilities.
    Returns: dict with baseline and best-smoothed metrics and the winning grid point."""
    vm = P["m_va"] == 1
    em = P["m_expr"] == 1
    vnf = list(P["videoname_frames"])

    # ---- baseline (no smoothing) ----
    cV0 = CCC_score(P["y_va"][vm, 0], P["va_pred"][vm, 0])
    cA0 = CCC_score(P["y_va"][vm, 1], P["va_pred"][vm, 1])
    expr_probs = _softmax(P["expr_logits"].astype(np.float64))
    yhat0 = expr_probs.argmax(-1)
    f10, _, _ = metric_for_Exp(P["y_expr"][em], yhat0[em])
    base = {"ccc_V": cV0, "ccc_A": cA0, "F1_EXPR": f10}

    # ---- grid-search VA ----
    best_va = (-np.inf, None, None, cV0, cA0)
    for s in SIGMAS:
        for d in DELTAS:
            sm = gaussian_smooth_per_video(P["va_pred"], vnf, sigma=s, delta=d)
            cV = CCC_score(P["y_va"][vm, 0], sm[vm, 0])
            cA = CCC_score(P["y_va"][vm, 1], sm[vm, 1])
            score = 0.5 * (cV + cA)
            if score > best_va[0]:
                best_va = (score, s, d, cV, cA)

    # ---- grid-search EXPR probabilities ----
    best_expr = (-np.inf, None, None, f10)
    for s in SIGMAS:
        for d in DELTAS:
            sm = gaussian_smooth_per_video(expr_probs, vnf, sigma=s, delta=d)
            f1, _, _ = metric_for_Exp(P["y_expr"][em], sm.argmax(-1)[em])
            if f1 > best_expr[0]:
                best_expr = (f1, s, d, f1)

    _, sV, dV, cV_b, cA_b = best_va
    f1_b, sE, dE, _ = best_expr
    return {
        "variant": P["variant"],
        "baseline": base,
        "smoothed": {"ccc_V": cV_b, "ccc_A": cA_b, "F1_EXPR": f1_b},
        "best_va_grid":   {"sigma": sV, "delta": dV},
        "best_expr_grid": {"sigma": sE, "delta": dE},
        "P_MTL_base":     base["ccc_V"] + base["ccc_A"] + base["F1_EXPR"],
        "P_MTL_smoothed": cV_b + cA_b + f1_b,
    }

## Run on four representative variants

Pick the visual unimodal plus three fusions spanning the Pareto front:
`visual_only` (anchor), `f1_concat` (simplest fusion), `f3_gate`
(mid-Pareto), `f4_xattn` (CREMA champion).

In [4]:
VARIANTS = [
    "crema_visual_only",
    "crema_f1_concat",
    "crema_f3_gate",
    "crema_f4_xattn",
]

rows = []
for name in VARIANTS:
    ck = RESULTS / name / "best.pt"
    if not ck.exists():
        print(f"[skip] {name}: no checkpoint at {ck}")
        continue
    print(f"[run ] {name}")
    preds = collect_predictions(CONFIGS / f"{name}.yaml", ck)
    r = evaluate_smoothing(preds)
    rows.append({
        "variant": name,
        "ccc_V_base":  r["baseline"]["ccc_V"],
        "ccc_A_base":  r["baseline"]["ccc_A"],
        "F1_EXPR_base": r["baseline"]["F1_EXPR"],
        "P_MTL_base":   r["P_MTL_base"],
        "ccc_V_sm":    r["smoothed"]["ccc_V"],
        "ccc_A_sm":    r["smoothed"]["ccc_A"],
        "F1_EXPR_sm":  r["smoothed"]["F1_EXPR"],
        "P_MTL_sm":    r["P_MTL_smoothed"],
        "delta_P_MTL": r["P_MTL_smoothed"] - r["P_MTL_base"],
        "sigma_VA":    r["best_va_grid"]["sigma"],
        "delta_VA":    r["best_va_grid"]["delta"],
        "sigma_EXPR":  r["best_expr_grid"]["sigma"],
        "delta_EXPR":  r["best_expr_grid"]["delta"],
    })

df = pd.DataFrame(rows)
df

[run ] crema_visual_only


c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.c

[run ] crema_f1_concat


c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.c

[run ] crema_f3_gate


c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.c

[run ] crema_f4_xattn


c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.c

,variant,ccc_V_base,ccc_A_base,F1_EXPR_base,P_MTL_base,ccc_V_sm,ccc_A_sm,F1_EXPR_sm,P_MTL_sm,delta_P_MTL,sigma_VA,delta_VA,sigma_EXPR,delta_EXPR
0,crema_visual_only,0.682083,0.298906,0.379893,1.360882,0.751364,0.350519,0.445264,1.547146,0.186264,10,50,50,50
1,crema_f1_concat,0.740063,0.452515,0.438914,1.631492,0.827966,0.541095,0.522074,1.891135,0.259643,100000,50,100,50
2,crema_f3_gate,0.735184,0.514094,0.427045,1.676324,0.809522,0.584824,0.495554,1.889900,0.213576,100000,50,500,10
3,crema_f4_xattn,0.833264,0.539012,0.488603,1.860879,0.853336,0.557574,0.501199,1.912109,0.051230,100000,50,100,50


In [5]:
out = RESULTS / "ablation_smoothing.md"
with out.open("w", encoding="utf-8") as fh:
    fh.write("# Per-video smoothing ablation (CREMA-D)\n\n")
    fh.write(
        "Paper A's smoothing grid (sigma, delta) applied post-hoc at evaluation "
        "time to V/A regressions and EXPR probabilities. AU is NOT smoothed "
        "(cell 66 of Paper A confirmed it hurts AU F1). P_MTL here is the "
        "three-task variant used throughout the interim: CCC_V + CCC_A + F1_EXPR.\n\n"
    )
    fh.write(df.to_markdown(index=False, floatfmt=".4f"))
    fh.write("\n")
print("wrote", out)

# Append to ablations_summary.md if present.
summary = RESULTS / "ablations_summary.md"
if summary.exists():
    with summary.open("a", encoding="utf-8") as fh:
        fh.write("\n## Per-video smoothing (CREMA-D)\n\n")
        fh.write(df.to_markdown(index=False, floatfmt=".4f"))
        fh.write("\n")
    print("appended to", summary)

wrote C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\results\interim\ablation_smoothing.md
appended to C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\results\interim\ablations_summary.md
